In [1]:
pip install langgraph-checkpoint-postgres psycopg[binary,pool]

  Using cached tzdata-2026.2-py2.py3-none-any.whl.metadata (1.4 kB)
   ---------------------------------------- 0.0/49.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/49.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/49.0 kB ? eta -:--:--
   ------------------------- -------------- 30.7/49.0 kB 435.7 kB/s eta 0:00:01
   ---------------------------------------- 49.0/49.0 kB 496.8 kB/s eta 0:00:00
   ---------------------------------------- 0.0/3.6 MB ? eta -:--:--
   - -------------------------------------- 0.1/3.6 MB 2.6 MB/s eta 0:00:02
   ------- -------------------------------- 0.7/3.6 MB 8.4 MB/s eta 0:00:01
   --------------------- ------------------ 1.9/3.6 MB 15.1 MB/s eta 0:00:01
   ------------------------------------ --- 3.3/3.6 MB 18.9 MB/s eta 0:00:01
   -------------------------------------- - 3.5/3.6 MB 15.9 MB/s eta 0:00:01
   ---------------------------------------- 3.6/3.6 MB 13.4 MB/s eta 0:00:00
   ------------------------------


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
from langgraph.graph import StateGraph,START, MessagesState
from langchain_groq import ChatGroq
from langgraph.checkpoint.postgres import PostgresSaver
from dotenv import load_dotenv
load_dotenv()
import os

In [9]:
llm=ChatGroq(model="openai/gpt-oss-120b")

In [10]:
def call_model(state: MessagesState):
    response = llm.invoke(state["messages"])
    return { "messages": [response]}

In [11]:
builder = StateGraph(MessagesState)
builder.add_node("call_model", call_model)
builder.add_edge(START,"call_model")


In [12]:
DB_URL= os.getenv("DB_URL")

In [13]:
with PostgresSaver.from_conn_string(DB_URL) as checkpointer:
    checkpointer.setup()
    graph=builder.compile(checkpointer=checkpointer)
    t1 = {"configurable":{"thread_id":"thread_1"}}
    graph.invoke({"messages":[{"role":"user","content":"Hi my name is nitish"}]},t1)
    out1 = graph.invoke({"messages":[{"role":"user","content":"what is my namw"}]},t1)
    print("thread_1",out1["messages"][-1].content)


thread_1 Your name is **Nitish**. If there’s anything else you’d like to discuss or ask about, just let me know!
